In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import contextily as cx
from adjustText import adjust_text
import itertools


from neuralhydrology.nh_run import start_run, eval_run, finetune
from neuralhydrology.nh_run import continue_run
from neuralhydrology.utils.config import Config
from neuralhydrology.evaluation import get_tester, metrics

In [2]:
# -------- Paths ---------
CONFIG_PATH = Path("./basins_subset.yml")
RUNS_DIR = Path("runs")
ATTRIBUTES_FILE= Path('../extended_dataset/data/attributes/attributes_other.csv')

In [3]:
# precip_products = [
#     # 'total_precipitation_sum'
#     # "chirps_precipitation",
#     "mswep_precipitation"
# ]

In [ ]:
# precip_products = [
#     # "total_precipitation_sum",
#     # "chirps_precipitation",
#     # "mswep_precipitation",
#     # "camels_precipitation",
#     "chirps_precipitation"
# ]

seeds = [333, 444, 555, 666, 777, 888] #  

base_non_precip_inputs = [
    "temperature_2m_max",
    "temperature_2m_min",
    "surface_net_solar_radiation_mean",
]

with open(CONFIG_PATH, "r") as f:
    base_config = yaml.safe_load(f)

use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()

# for r in range(1, len(precip_products) + 1):

precip_combos = [
    # ("camels_precipitation",),
    ("total_precipitation_sum",),
    # ("chirps_precipitation",),
    # ("mswep_precipitation",),
]

# for precip_combo in itertools.combinations(precip_products, r):
for precip_combo in precip_combos:
    for seed in seeds:
        config = base_config.copy()

        config["dynamic_inputs"] = [*base_non_precip_inputs, *precip_combo]
        config["seed"] = seed

        precip_name = "_".join(precip_combo)
        config["experiment_name"] = (
            f"{precip_name}_seq_{config['seq_length']}"
            f"_{config['predict_last_n']}_epochs_{config['epochs']}"
            f"_hidden_{config['hidden_size']}"
            f"_dropout_{str(config['output_dropout']).replace('.', '')}"
            f"_fb_{config['initial_forget_bias']}"
            f"_seed{seed}"
        )

        temp_config_path = Path(f"temp_{precip_name}_seed{seed}.yml")
        with open(temp_config_path, "w") as f:
            yaml.dump(config, f)

        print(f"Running: {config['experiment_name']}")

        if use_gpu:
            start_run(config_file=temp_config_path)
        else:
            start_run(config_file=temp_config_path, gpu=-1)

        temp_config_path.unlink()  # clean up temp file after run

Running: total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333
2026-04-27 21:44:53,089: Logging to /home/azureuser/sky_workdir/extending_caravan/157_basins_runs/runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_2704_214453/output.log initialized.
2026-04-27 21:44:53,090: ### Folder structure created at /home/azureuser/sky_workdir/extending_caravan/157_basins_runs/runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_2704_214453
2026-04-27 21:44:53,090: ### Run configurations for total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333
2026-04-27 21:44:53,091: batch_size: 256
2026-04-27 21:44:53,091: clip_gradient_norm: 1
2026-04-27 21:44:53,092: data_dir: ../extended_dataset/data
2026-04-27 21:44:53,092: dataset: generic
2026-04-27 21:44:53,093: device: cuda:0
2026-04-27 21:44:53,093: dynamic_inputs: ['temperature_2m_max', 'temperature_2m_min', 'surface_net_solar_radiation_mea

In [13]:
# base_config_path = Path("basins_excluded.yml")

# with open(base_config_path, "r") as f:
#     base_config = yaml.safe_load(f)

In [7]:
# for r in range(1, len(precip_products) + 1):
#     for precip_combo in itertools.combinations(precip_products, r):
#         config = base_config.copy()

#         # Update dynamic inputs:
#         # Keep temperature and radiation, replace precip
#         config["dynamic_inputs"] = [
#             "surface_net_solar_radiation_mean",
#             "temperature_2m_max",
#             "temperature_2m_min",
#             *precip_combo
#         ]

#         # Create unique experiment name
#         precip_name = "_".join(precip_combo)
#         config["experiment_name"] = (
#             f"precip_{precip_name}"
#         )

#         # Save temporary config file
#         temp_config_path = Path(f"temp_{precip_name}")
#         with open(temp_config_path, "w") as f:
#             yaml.dump(config, f)

#         print(f"Running: {config['experiment_name']}")

#         # Run training
#         if torch.cuda.is_available() or torch.backends.mps.is_available():
#             start_run(config_file=temp_config_path)
#         else:
#             start_run(config_file=temp_config_path, gpu=-1)

In [1]:
# # by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
# if torch.cuda.is_available() or torch.backends.mps.is_available():
#     start_run(config_file=Path("basins_excluded.yml"))

# # fall back to CPU-only mode
# else:
#     start_run(config_file=Path("basins_excluded.yml"), gpu=-1)